# 01 — Data Inventory

**Influenza Season Forecasting** · Notebook 1 of 5

**Purpose:** Load every raw data source, audit its structure, coverage, and missingness, and record the loading quirks and project decisions. This notebook produces *no model and no cleaned data* — it is the auditable record of what we have and how to read it. Cleaning and target construction happen in `02_cleaning.ipynb`.

**Scope decisions (confirm with advisor):**
- **National only.** ILINet is exported at the National level (`% WEIGHTED ILI` is the target). Matches the template's ~1,100-row target.
- **Option B feature strategy.** Core features (ILI%, strain) use the full ~22 seasons (2003-04 →). Hospitalization and vaccine features only exist from 2009 on, so they are treated as *optional enrichment evaluated on the 2009+ subset*, not core inputs.
- **WHO FluNet excluded.** See section 5.
- **Pandemic seasons (2009 H1N1, 2020-21) held out** as labeled special cases downstream, not dropped here.


## Setup

In [ ]:
from pathlib import Path
import pandas as pd

# Point this at wherever the raw files live (gitignored). In Colab, upload here or mount Drive.
DATA_DIR = Path("data/raw")

FILES = {
    "ilinet":       "ILINet.csv",
    "nrevss_pre":   "ICL_NREVSS_Combined_prior_to_2015_16.csv",
    "nrevss_phl":   "ICL_NREVSS_Public_Health_Labs.csv",
    "nrevss_clin":  "ICL_NREVSS_Clinical_Labs.csv",
    "flusurv":      "FluSurveillance_Custom_Download_Data.csv",
    "fluvax":       "FluVaxView.csv",   # renamed from the long CDC default filename
}

missing = [f for f in FILES.values() if not (DATA_DIR / f).exists()]
if missing:
    print("Place these files in", DATA_DIR.resolve(), "before running:")
    for f in missing: print("  -", f)
else:
    print("All source files found in", DATA_DIR.resolve())


## 1. ILINet — core ILI% series (2003+)

Outpatient influenza-like illness. This is the spine of the project; everything aligns to it.

**Loading quirks:**
- **Line 1 is a title sentence**, not the header → `skiprows=1`.
- **Missing values are the sentinel `X`**, not blanks → `na_values=["X"]`. They appear in the age-stratified columns and the REGION field, *not* in the target.
- **Target column:** `% WEIGHTED ILI` (population-weighted). Use this, not `%UNWEIGHTED ILI`.


In [ ]:
ili = pd.read_csv(DATA_DIR / FILES["ilinet"], skiprows=1, na_values=["X"])

print("shape:", ili.shape)
print("region types:", ili["REGION TYPE"].unique().tolist())
print("calendar year range:", ili["YEAR"].min(), "->", ili["YEAR"].max())
print("target '% WEIGHTED ILI' — missing:", int(ili["% WEIGHTED ILI"].isna().sum()),
      "| range: {:.2f} to {:.2f}".format(ili["% WEIGHTED ILI"].min(), ili["% WEIGHTED ILI"].max()))
print("\ncolumns:")
for c in ili.columns: print("  -", c)
ili.head(3)


## 2. NREVSS virological — strain composition, core (2003+)

US domestic lab-confirmed strain breakdown. This powers the `dominant_strain` feature, and it covers 2003+, so it replaces WHO FluNet for our purposes.

**Critical quirk — the 2015-16 reporting break.** CDC restructured virologic reporting in 2015-16:
- **Pre-2015-16:** the *Combined* file carries the full subtype breakdown (A/H1, A/H3, 2009 H1N1, B).
- **2015-16 onward:** clinical labs *stopped subtyping* (Clinical Labs file has only % A / % B); the subtype detail moved to the *Public Health Labs* file.

So to build a continuous `dominant_strain` series across all seasons, `02_cleaning.ipynb` must **stitch Combined (pre-2015-16) + Public Health Labs (2015-16 on)**. The Clinical Labs file is for overall positivity, not subtype. We only *inventory* the three files here; the stitch happens in cleaning.


In [ ]:
nrevss = {}
for key, label in [("nrevss_pre","Combined (pre-2015-16, has subtype)"),
                   ("nrevss_phl","Public Health Labs (2015-16+, has subtype)"),
                   ("nrevss_clin","Clinical Labs (2015-16+, positivity only — NOT subtype)")]:
    d = pd.read_csv(DATA_DIR / FILES[key], skiprows=1, na_values=["X","XX"])
    nrevss[key] = d
    print(f"{label}")
    print(f"   rows={len(d)}  years={d['YEAR'].min()}-{d['YEAR'].max()}")
    print(f"   cols: {list(d.columns)}\n")


## 3. FluSurv-NET — hospitalization rates (enrichment, 2009+ · Option B)

Lab-confirmed influenza hospitalization rates per 100,000. **Starts 2009-10**, so under Option B this is enrichment on the 2009+ subset, not a core feature.

**Loading quirks:**
- **Two preamble lines** (title + CDC attribution) → `skiprows=2`.
- **A disclaimer footer** parses as junk rows → filter to real catchments (`CATCHMENT == "Entire Network"`) to drop them.
- **Missing sentinel is `null`** → `na_values=["null"]`.
- Stratified by age/sex/race/virus-type → filter all four to `"Overall"` for the aggregate weekly series. Key columns: `WEEKLY RATE`, `CUMULATIVE RATE`.
- Note: two `YEAR` columns (season label + calendar year); pandas renames the second `YEAR.1`.


In [ ]:
raw = pd.read_csv(DATA_DIR / FILES["flusurv"], skiprows=2, na_values=["null"])
raw.columns = [c.strip() for c in raw.columns]

hosp = raw[raw["CATCHMENT"] == "Entire Network"].copy()          # drops the disclaimer footer rows
hosp_overall = hosp[(hosp["AGE CATEGORY"]=="Overall") & (hosp["SEX CATEGORY"]=="Overall") &
                    (hosp["RACE CATEGORY"]=="Overall") & (hosp["VIRUS TYPE CATEGORY"]=="Overall")]

print("aggregate weekly rows:", len(hosp_overall))
print("season range:", hosp_overall["YEAR"].min(), "->", hosp_overall["YEAR"].max())
print("WEEKLY RATE missing:", int(hosp_overall["WEEKLY RATE"].isna().sum()))
hosp_overall[["YEAR","WEEK","WEEKLY RATE","CUMULATIVE RATE"]].head(3)


## 4. FluVaxView — vaccine coverage (enrichment, 2009+ · Option B)

Cumulative seasonal flu vaccine uptake. **Starts 2009-10** → enrichment under Option B.

**Extraction recipe (national, all-ages, season-end):**
- Filter `Geography == "United States"`, `Dimension Type == "Age"`.
- All-ages label is **`>=6 Months`** in older seasons but **`Greater than 6 Months flu`** in recent ones (2023-24+). Accept both, or 2023-24 silently drops out.
- Coverage is **cumulative and monotonic within a season**, so the season-end value is the **max** over the season's months. (Do *not* take the highest month number — the season-end months Jan–May sort numerically below December.)
- 30MB file → read in chunks.


In [ ]:
ALLAGES = [">=6 Months", "Greater than 6 Months flu"]

keep = []
for chunk in pd.read_csv(DATA_DIR / FILES["fluvax"], chunksize=200_000, dtype=str):
    m = chunk[(chunk["Geography"]=="United States") &
              (chunk["Dimension Type"]=="Age") &
              (chunk["Dimension"].isin(ALLAGES))]
    if len(m): keep.append(m)

vax = pd.concat(keep)
vax["Estimate (%)"] = pd.to_numeric(vax["Estimate (%)"], errors="coerce")

# monotonic cumulative within season -> season-end coverage = max
season_cov = (vax.dropna(subset=["Estimate (%)"])
                 .groupby("Season/Survey Year")["Estimate (%)"].max()
                 .rename("season_end_coverage_pct"))

print("national all-ages seasons:", len(season_cov),
      "| range: {:.1f}% to {:.1f}%".format(season_cov.min(), season_cov.max()))
season_cov.to_frame()


## 5. WHO FluNet — EXCLUDED

The uploaded FluNet export (`DataExport` sheet) is **not used** in this project. Two reasons:

1. **Coverage doesn't reach the study period.** It spans only **2022–2026** (184 countries, ~52k rows), with no data before 2022. The task requires 2003 onward.
2. **Redundant.** It's global; the US slice is ~428 rows. US strain composition back to 2003 is already covered, better, by the NREVSS files in section 2.

Documented here so the exclusion is a recorded decision, not an omission.


## Coverage summary & next step

| Source | Role | Seasons | Notes |
|---|---|---|---|
| ILINet (`% WEIGHTED ILI`) | **Core target** | ~22 (2003-04 →) | clean; skiprows=1, `X` sentinel |
| NREVSS strain | **Core feature** | ~22 (2003+) | stitch Combined + PHL across 2015-16 break |
| FluSurv-NET hosp | Enrichment (Option B) | 16 (2009-10 →) | skiprows=2, drop footer, `null` sentinel |
| FluVaxView coverage | Enrichment (Option B) | 16 (2009-10 →) | dual all-ages label; season = max |
| WHO FluNet | **Excluded** | — | 2022+ only; redundant |

**Next:** `02_cleaning.ipynb` — align ILINet to MMWR season weeks (40→39), build the `peak_ili_pct` and `peak_week` targets per season, verify each peak sits in the season interior (not pinned to a boundary), and perform the NREVSS strain stitch.


In [ ]:
# Optional: write a machine-readable coverage manifest for downstream notebooks
import json
manifest = {
    "national_only": True,
    "feature_strategy": "Option B (core 2003+, enrichment 2009+)",
    "excluded_sources": ["WHO FluNet (2022+ only, redundant with NREVSS)"],
    "ilinet_target_col": "% WEIGHTED ILI",
    "nrevss_stitch": "Combined(pre-2015-16) + PublicHealthLabs(2015-16+)",
}
print(json.dumps(manifest, indent=2))
